# 01 · EDA — HackSpain 2026 · Reto X Ray (Embat)

Notebook vivo de exploración. Cada sección: **código → conclusión**. Las conclusiones van en celdas markdown para poder consultarlas sin re-ejecutar.

**Requisitos:** `pandas` (y `jupyter` para abrirlo). Los CSV están en `../data/`.

In [ ]:
import pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
D = "../data/"

companies = pd.read_csv(D+"companies.csv", parse_dates=["created_at"])
groups    = pd.read_csv(D+"groups.csv")
debt      = pd.read_csv(D+"debt_products.csv")
bank      = pd.read_csv(D+"banking_products.csv")
balances  = pd.read_csv(D+"balances.csv")
tx = pd.read_csv(D+"transactions.csv", parse_dates=["date"],
                 usecols=["company_id","product_id","date","amount","exchange_rate","status","accounting_status","category","counterparty_id"])
inv = pd.read_csv(D+"invoices.csv", parse_dates=["issuance_date","due_date","payment_date"],
                  usecols=["operation_id","company_id","document_type","issuance_date","due_date","payment_date","amount","pending_amount","currency","accounting_currency","exchange_rate","status","counterparty_id"])
cur = companies.set_index("company_id").currency

## 1. Primeras conclusiones (sesión 2026-09-19)

### Cobertura
- `transactions`: 2,56 M filas, **1.286 empresas** (todas). `invoices`: 898 k filas, solo **785 empresas** → 501 sin facturas.
- La historia **no es de 24 meses para todas**: ~25 % de empresas empieza después de nov-2025 y el volumen mensual de transacciones se multiplica ×4 entre sep-2024 y jul-2026 → panel con longitud desigual.
- **Sep-2026 es un mes parcial** (1 día de datos): excluir o marcar.
- Moneda de la empresa: EUR en 1.149; el resto repartido en 27 monedas.

### Calidad de datos
- **Outliers extremos en importes**: transacciones de ±3·10⁹; ~9.400 filas > 10 M, concentradas en 95 empresas. Medias/desviaciones estándar no valen → tratamiento robusto (winsorizar p99,9 o mediana/MAD).
- **`exchange_rate` (CORREGIDO, ver sección 16)**: en una primera lectura se consideró no fiable; era un error de comparación. El importe está en la moneda de la *cuenta* y `exchange_rate` es el cambio a la moneda de la empresa (`importe / tasa`).
- **`payment_date` engaña**: nunca es nulo y en facturas `overdue`/`pending` coincide con `due_date` en 96–98 % de los casos. → La morosidad se saca de **`status`**, no de comparar fechas.
- Signo del importe en facturas: solo 41 % positivos. Hipótesis a validar: positivo = venta (a cobrar), negativo = compra (a pagar).
- `category` de transacciones: 25 % con valor `-`. Con señal clara: `collection`, `payment`, `salary`, `tax`, `debt_repayment`.
- `debt_products` **no tiene fechas** → contexto estático por empresa; no meterlo en el histórico mensual (fuga de futuro).

### Decisiones pendientes
1. Signo de facturas (positivo = venta).
2. Outliers: winsorizar p99,9 vs. excluir las 95 empresas.

### Panel propuesto `company_id × month`
Caja (entradas, salidas, flujo neto, saldo reconstruido hacia atrás desde `balances.csv`) · Facturas (% vencido por `status`, DSO, concentración) · Deuda (estática, aparte) · Calidad (meses de historia, flag mes parcial).

## 2. Cobertura y longitud de historia

In [ ]:
print("tx:", tx.shape, "| empresas:", tx.company_id.nunique())
print("inv:", inv.shape, "| empresas:", inv.company_id.nunique())
print("rango tx:", tx.date.min(), "→", tx.date.max())
tx.groupby(tx.date.dt.to_period("M")).size().rename("n_tx").to_frame().T

In [ ]:
h = tx.groupby("company_id").date.agg(first="min", last="max", n="count")
h["meses"] = ((h["last"] - h["first"]).dt.days / 30.4).round()
print(h.meses.describe())
print("empresas que empiezan tras 2025-11-01:", (h["first"] > "2025-11-01").mean().round(3))

## 3. Moneda y tipo de cambio

> Nota: la conclusión inicial de esta sección (tasa no fiable) fue errónea. Ver sección 16.

In [ ]:
t = tx.assign(cur=tx.company_id.map(cur))
print(companies.currency.value_counts().head(8).to_dict())
t.groupby("cur").exchange_rate.describe().round(2).sort_values("count", ascending=False).head(10)

In [ ]:
i = inv.assign(ccur=inv.company_id.map(cur))
print("moneda factura == moneda empresa:", (i.currency==i.ccur).mean().round(3))
print("moneda contable == moneda empresa:", (i.accounting_currency==i.ccur).mean().round(3))

## 4. Outliers de importe

In [ ]:
print(tx.amount.abs().quantile([.5,.9,.99,.999,.9999]))
big = tx[tx.amount.abs()>1e7]
print("filas >10M:", len(big), "| empresas:", big.company_id.nunique())

## 5. Facturas: status, signo y `payment_date`

In [ ]:
print(inv.status.value_counts())
print(inv.document_type.value_counts())
print("% importes positivos:", (inv.amount>0).mean().round(3))
print("payment_date nulo:", inv.payment_date.isna().mean())
(inv.payment_date==inv.due_date).groupby(inv.status).mean().round(2)

## 6. Categorías de transacciones

In [ ]:
print(tx.category.value_counts())
print(tx.status.value_counts())

## 7. Próximos pasos
- Resolver las dos decisiones pendientes y construir el panel mensual (`company_id × month`).
- Reconstruir el saldo mensual hacia atrás desde `balances.csv` (filtrar solo cuentas de `banking_products`).
- Deuda: explorar `granted/outstanding/liquidity` y `debt_schedule_config` por empresa.
- Grupos: comprobar cuánto se parecen las empresas de un mismo grupo (para decidir el split por `group_id`).

## 8. Decisiones tomadas (2026-09-19)

1. **Signo de facturas: positivo = venta (a cobrar), negativo = compra (a pagar).** Verificado cruzando contrapartes con transacciones: 88 % de facturas positivas van a contrapartes con `collection`; 94 % de las negativas a contrapartes con `payment`.
2. **Outliers: winsorizar importes al p99,9** (no excluir empresas; el test oculto puede traer otras igual de raras y hay que puntuarlas todas). Casi todas las features serán ratios.
3. **El cash flow sale de `transactions` (1.286 empresas), no de facturas.** Las facturas solo alimentan el pilar de capital circulante.
4. **Datos faltantes:** sin facturas (501 empresas, 39 %) → pilar vacío y pesos renormalizados, sin imputar ceros; columna `pilares_disponibles` como confianza. Sin deuda → caso real, carga mínima (buena), no dato faltante.

In [ ]:
inv_c = inv.dropna(subset=["counterparty_id"]); tx_c = tx.dropna(subset=["counterparty_id"])
a = tx_c[tx_c.category.isin(["collection","payment"])].groupby(["company_id","counterparty_id","category"]).amount.sum().reset_index()
b = inv_c.groupby(["company_id","counterparty_id"]).amount.sum().rename("inv").reset_index()
a.merge(b, on=["company_id","counterparty_id"]).groupby("category").apply(lambda d: (d.inv>0).mean()).round(3)

## 9. Sesgo de las empresas sin facturas (501 empresas)

La ausencia de facturas **no es aleatoria: depende del ERP**.

| | Sin facturas (501) | Con facturas (785) |
|---|---|---|
| Tienen ERP | 8 % | 90 % |
| Meses de historia (mediana) | 16 | 20 |
| Entradas totales (mediana) | 6,3 M | 4,0 M |
| Importe mediano por movimiento | 918 | 618 |
| Nº transacciones (mediana) | 871 | 825 |

- Casi todos los ERP tienen 93–100 % de cobertura de facturas; sin ERP solo 14 %.
- No es un problema de calidad: tienen tantas transacciones como las demás. Tienden a ser más grandes y con menos historia.
- Los grupos se parecen: 33 % de grupos sin ninguna empresa con facturas, 51 % con todas, 16 % mixtos → respalda el split por `group_id`.
- **Implicaciones:** renormalizar pesos sigue siendo correcto; comprobar que el score no correlaciona con "tener ERP"; validar por separado en ambos subconjuntos; estimar cobros desde transacciones queda como mejora futura.

In [ ]:
co = companies.assign(hasinv=companies.company_id.isin(set(inv.company_id)))
co["erp_known"] = co.erp.notna()
print(co.groupby("hasinv").erp_known.mean().round(2))
gg = co.groupby("group_id").hasinv.mean()
print("grupos todo-sin:", (gg==0).mean().round(2), "| todo-con:", (gg==1).mean().round(2))

## 10. Reconstrucción del saldo (liquidez)

Objetivo: `saldo(t) = saldo_final (balances.csv) − Σ transacciones posteriores a t`.

Hallazgos:
- **Solo cuentas de `banking_products`**: `balances.csv` mezcla 5.760 cuentas de banco, 2.207 de deuda (saldo negativo) y 29 sin identificar. Sumar todo mezclaría caja con deuda.
- **El 7,2 % de las transacciones está en productos de deuda** (`debt_products`), no en cuentas de banco. Para el flujo de caja hay que decidir si se excluyen o se tratan aparte (amortizaciones, disposiciones de línea).
- **Outliers también en saldos**: cuentas `checking` con saldo de ±10⁹ y hasta 10¹¹. Excluir |saldo| > 10⁸ o winsorizar.
- (Los importes de esta sección se recalcularon con conversión de moneda en el panel final, ver sección 16.)
- **Cobertura:** 1.269/1.286 empresas tienen algún saldo bancario (las 17 restantes no tendrán liquidez).
- **Calidad de la reconstrucción (nivel empresa, tras winsorizar):** el saldo de apertura implícito es positivo en el 91 % de las empresas. **Pero el 17 % (218 empresas) pasa a saldo negativo en algún mes**, señal de que el histórico de transacciones no cuadra con el saldo final para ellas (movimientos incompletos o cuentas no incluidas). En la mayoría el descuadre es pequeño (mediana ≈ el propio saldo), pero hay casos extremos.
- Solo 0,27 % de las transacciones son `pending`: no afecta.
- **Decisión para el diseño:** la liquidez absoluta reconstruida es fiable para ~83 % de las empresas. Para el resto, marcar un flag `saldo_inconsistente` y apoyarse en ratios de flujo (`runway`, cobertura de salidas) en vez del nivel.

In [ ]:
bk = bank.product_id
bal["kind"] = bal.product_id.map(lambda p: "bank" if p in set(bk) else "otro")
print(bal.kind.value_counts())
tx_b = tx[tx.product_id.isin(bk)]
print("% tx en cuentas de banco:", len(tx_b)/len(tx))
bb = bal[(bal.kind=="bank") & (bal.balance.abs()<1e8)]
cb = bb.groupby("company_id").balance.sum()
tx_b = tx_b.assign(amount=tx_b.amount.clip(*tx_b.amount.quantile([.001,.999])), m=tx_b.date.dt.to_period("M"))
mo = tx_b.groupby(["company_id","m"]).amount.sum().reset_index()
mo = mo[mo.m < "2026-09"]
opening = cb - mo.groupby("company_id").amount.sum()
mo = mo.merge(opening.rename("open"), left_on="company_id", right_index=True).sort_values(["company_id","m"])
mo["lvl"] = mo.open + mo.groupby("company_id").amount.cumsum()
print("apertura<0:", (opening<0).mean().round(3), "| empresas con saldo<0 alguna vez:", (mo.groupby("company_id").lvl.min()<0).mean().round(3))

## 11. Deuda (`debt_products`, `debt_schedule_config`)

- **Cobertura:** 2.239 productos en solo **378 empresas (29 %)**. Tipos: loan 1.022, lineofcredit 536, confirming 229, leasing 179, guarantee 155, mortgage 60, renting 34, factoring 24. La mayoría de empresas con deuda tiene 1–3 productos.
- **Convención de signos:** `granted` y `outstanding` son **negativos** (91 % y 60 %; lo que se debe). `liquidity` es siempre positiva (disponible). `outstanding` = 0 en el 33 % (línea sin usar o deuda saldada). Un 6,5 % de `outstanding` es positivo (probable saldo a favor).
- **Nulos:** `granted` 7,5 %, `liquidity` 64 % (solo relevante en líneas de crédito/confirming/factoring; 86–100 % nulo en loan, leasing, mortgage, renting).
- **Utilización `outstanding/granted`:** mediana 0,35, P75 0,83; >1 en solo 0,9 %; negativa en 6,4 % (signos incoherentes) → recortar a [0,1] y descartar `granted == 0`.
- **Outliers:** granted/outstanding hasta ±3·10⁸ → winsorizar / usar ratios.
- **Sin dimensión temporal:** `created_at` (2022–2026, 75 % desde 2025) es cuándo se **conectó** el producto, no cuándo se contrató la deuda. No permite reconstruir la deuda histórica → carga de deuda estática por empresa.
- **`debt_schedule_config`:** solo 87 filas (3,9 % de los productos), todas `constant quote`, 58 fijos / 29 variables, tipo mediano 3 %. Sirve para enriquecer, no como pilar base.
- **⚠ El fichero de deuda está incompleto:** 525 empresas tienen transacciones `debt_repayment`, pero **241 de ellas no tienen ningún producto en `debt_products`**. Además 94 empresas con deuda no tienen ningún `debt_repayment`. Por tanto "sin `debt_products`" **no** implica "sin deuda".
- **Decisión:** el pilar de deuda se construye con dos señales: (a) estática, `outstanding/granted` y `outstanding/entradas anuales` de `debt_products`; (b) dinámica y mensual, flujo `debt_repayment` de transacciones sobre entradas (servicio de deuda), que cubre a las 525 empresas y sí tiene fecha. La (b) resuelve tanto el problema de la fuga temporal como la incompletitud.

In [ ]:
d = debt.copy()
print("empresas con deuda:", d.company_id.nunique())
print(d[["granted","outstanding","liquidity"]].isna().mean().round(3))
a = d.dropna(subset=["granted","outstanding"]); a = a[a.granted != 0]
print((a.outstanding / a.granted).describe().round(2))
dr = set(tx[tx.category=="debt_repayment"].company_id)
print("con debt_repayment:", len(dr), "| sin debt_product:", len(dr - set(d.company_id)))

## 12. Estacionalidad y volatilidad del flujo de caja

Flujo mensual por empresa, solo cuentas de banco, importes winsorizados, sep-2026 excluido.

- **Estacionalidad agregada débil:** el índice de entradas por mes calendario (sobre la mediana de cada empresa) va de 0,90 a 1,13 (jul 1,13; ago 0,90; dic 1,06). El flujo neto sobre entradas es ≈0 casi todos los meses; solo enero es claramente negativo (−3,7 %).
- **Estacionalidad fuerte por categoría** (índice sobre la media anual): `tax` 0,5–1,7 (pico may/jul, valle mar/nov); `social_security` 1,5–1,9 de may a ago y ~0,5 de sep a dic; `salary` con pico de diciembre (1,52, paga extra) y valle sep/oct (0,6); `debt_repayment` 0,1–0,2 en sep–ene y 2,5–2,8 en may/jul.
- **⚠ Confusor:** cada mes calendario aparece 1 o 2 veces (sep–dic dos años; ene–ago también) y el número de empresas activas crece de 438 (sep-24) a ~1.200 (mar-26). Parte de la "estacionalidad" es cobertura de datos, no comportamiento. Las entradas medias por empresa suben de ~5,7 M a ~8,5 M en 2026 (salto ene–mar 2026, posible artefacto de incorporación).
- **Volatilidad:** CV mediano del flujo neto (desviación / entradas medias) = 0,42; P25 0,18, P75 0,93. Mediana de empresas con flujo neto negativo en el 50 % de los meses → **un mes negativo es normal y no es señal de deterioro** (base para distinguir bache de caída).
- **Persistencia:** autocorrelación del flujo neto relativo mes a mes = 0,25 (moderada): hay señal de tendencia, pero mucho ruido.
- **Huecos:** 9 % de empresas tienen meses sin transacciones dentro de su rango (2 % de los meses en total).
- **Longitud:** mediana de 18 meses con datos por empresa, 25 % con ≤10.
- **Decisiones:** (a) normalizar cada variable frente a la distribución de **todas** las empresas ese mismo mes (z robusto por mes): elimina estacionalidad común y deriva de cobertura; (b) medir el bache contra la volatilidad **propia** de cada empresa; (c) usar ventanas móviles de 3 meses para el flujo (un mes suelto es demasiado ruidoso); (d) exigir un mínimo de meses de historia para puntuar y marcar `historia_corta`.

## 13. Grupos y contrapartes

**Estructura de grupos**
- 250 grupos, media 5,1 empresas, mediana 3, máximo 22. El 28 % de los grupos tiene una sola empresa, pero **los grupos de ≥10 empresas concentran 640 de las 1.286 empresas (50 %)**.
- Todas las empresas de un grupo se dan de alta a la vez (dispersión de `created_at` mediana = 0 días): se incorporan como bloque.

**¿Se parecen las empresas de un mismo grupo?**
- Nivel: la varianza del flujo neto relativo medio entre empresas es 0,189 y dentro de un mismo grupo es 0,133 (ratio 0,70). El grupo explica **~30 %** de la varianza del nivel: efecto real, moderado.
- Dinámica: correlación temporal del flujo relativo mensual entre empresas del mismo grupo = 0,039 (mediana) frente a 0,005 entre grupos distintos. Casi nula: los grupos no se mueven al unísono mes a mes.
- **Conclusión:** el parecido es de *nivel*, no de *trayectoria*. Aun así, con ICC ≈ 0,3 y grupos de hasta 22 empresas, un split por `company_id` filtraría información → **split por `group_id`** (`GroupKFold`). Ojo: los grupos grandes desequilibran los folds; comprobar tamaños tras el reparto.

**Contrapartes**
- Solo el **9,8 % de las transacciones** tiene `counterparty_id` (mediana de 35 contrapartes por empresa).
- Solo el 0,2 % de contrapartes aparece en más de una empresa o grupo: **no hay red de proveedores/clientes compartidos ni intercompany detectable.** No se puede hacer análisis de contagio entre empresas.
- Concentración: el mayor cliente supone una mediana del 58 % de las entradas identificadas (P75 = 92 %; >50 % en el 60 % de empresas). **Sesgado**: solo cuenta entradas con contraparte resuelta, y la mayoría de movimientos no la tiene. Como pilar, la concentración es poco fiable → peso bajo o solo en empresas con buena cobertura de contraparte.

In [ ]:
co = companies.copy(); sz = co.groupby("group_id").size()
print("grupos de 1:", (sz==1).mean().round(2), "| empresas en grupos >=10:", sz[sz>=10].sum())
print("% tx con contraparte:", tx.counterparty_id.notna().mean().round(3))
cp = tx.dropna(subset=["counterparty_id"])
print("contrapartes en >1 empresa:", (cp.groupby("counterparty_id").company_id.nunique()>1).mean().round(4))

## 14. Trayectorias de ejemplo

Margen operativo a 3 meses (`Σ neto / Σ (entradas+salidas)`, acotado a ±1) de las 583 empresas con ≥20 meses de historia. Comparando los últimos 8 meses con los primeros 8:

- **Sí existen cambios de régimen claros:** 66 empresas mejoran (Δ > +0,3), 42 se deterioran (Δ < −0,3), 300 quedan estables (|Δ| < 0,1). El 19 % tiene |Δ| > 0,3. Mediana de Δ = 0: no hay tendencia global que confunda.
- **Se ven mejoras y deterioros sostenidos y de forma variada:** subidas escalonadas (COMP_0794, COMP_0049), caídas a ▁ y quedarse ahí (COMP_0390, COMP_0732), y empresas estables en nivel alto (COMP_0666) o bajo.
- **Baches:** una regla simple (mes 2σ bajo la mediana propia que se recupera) marca 202/583 (35 %): demasiado laxa, hay que endurecerla (≥2 meses fuera de banda, banda calibrada).
- **⚠ Cuidado con empresas casi inactivas:** hay empresas con entradas ≈ 0 (por ejemplo COMP_0325, COMP_0229) cuyo margen clavado en −1 no es "deterioro" sino cuenta dormida o solo de gastos. Usar `neto/(entradas+salidas)` en lugar de `neto/entradas` y marcar baja actividad.
- **Huecos de datos** visibles como espacios en la trayectoria (COMP_1054, COMP_1109).

In [ ]:
import numpy as np
panel = pd.read_csv("../data/processed/panel_monthly.csv")
wide = panel.pivot(index="month", columns="company_id", values="margin_3m")
import matplotlib.pyplot as plt
def plot(ids, title):
    fig, ax = plt.subplots(figsize=(9,3))
    for c in ids: ax.plot(wide.index, wide[c], label=c)
    ax.set_title(title); ax.tick_params(axis="x", rotation=90); ax.legend(fontsize=7); plt.show()
plot(["COMP_0794","COMP_0049","COMP_0598"], "Mejora sostenida")
plot(["COMP_0390","COMP_0732","COMP_0502"], "Deterioro sostenido")

## 15. Panel mensual (`scripts/build_panel.py`)

Salidas en `data/processed/`: `panel_monthly.csv` (22.172 filas, 1.282 empresas, sep-2024 a ago-2026) y `company_static.csv` (1.286 empresas).

**Columnas del panel:** `inflow_op`, `outflow_op`, `net_op`, `debt_service`, `transfer_net`, `n_tx`, `cash_end`, ventanas de 3 meses (`*_3m`), `margin_3m`, `runway_m`, `debt_service_ratio_3m`, `overdue_amt`, `sales_3m`, `pct_vencido`, `dso`, y flags `saldo_inconsistente`, `tiene_facturas`, `historia_corta`, `hist_months`.

**Decisiones:** solo cuentas de `banking_products`; winsorizado p0,1/p99,9; flujo operativo excluye `transfer`, `investment_*` y `debt_repayment` (transfer es casi unidireccional por empresa, 54 B entran y 34 B salen: no son traspasos internos que se cancelen); facturas de venta (`invoice`, `invoiceGroup`, importe > 0) con **vencido calculado históricamente** con `due_date` y `payment_date`, no con el `status` final.

**Validaciones:**
- El vencido histórico a ago-2026 coincide con el `status = overdue` del fichero (correlación 0,93; ratio mediano 1,00).
- `cash_end` final cuadra con `balances.csv` por construcción; 20 % de empresas marcadas `saldo_inconsistente`.
- Cobertura: 98 % con saldo, 56 % con facturas, 28 % con producto de deuda, 40 % paga deuda en el banco.

**Puntos débiles a vigilar:** `pct_vencido` supera 1 en el 22 % de filas (el vencido acumulado es mayor que 3 meses de ventas): conviene cambiar el denominador o normalizar; `runway_m` y `debt_service_ratio_3m` tienen colas extremas (usar z robusto); 4 empresas no tienen movimientos en cuentas de banco y no entran en el panel.

## 16. Corrección: tipo de cambio y contrapartes compartidas

**Tipo de cambio (rectificación).** Comparando la tasa con la moneda de la *cuenta* (no la de la empresa):
- El importe de `transactions` está en la moneda de la cuenta (`product_id`); `exchange_rate` lo convierte a la moneda de la empresa: `importe_empresa = importe / exchange_rate` (USD→EUR: 1,16; BRL: 6,22; CLP: 1.047).
- En cuentas EUR de empresas EUR la tasa es 1 en el 99,55 % de las filas. Las tasas "absurdas" que se vieron antes eran cuentas en BRL/CLP/etc.
- Comprobación de magnitud: cuentas CLP de empresas EUR, |importe| mediano 1.008.532 → dividido entre la tasa ≈ 984 EUR (cuentas EUR: 470).
- En `invoices`: `currency` es la de la factura, `accounting_currency` la de la empresa (coincide en el 97,7 %); misma división.
- Afecta a 259 empresas con alguna cuenta en otra moneda (142 con >10 % de sus transacciones, 74 con >50 %). `build_panel.py` ya convierte; los saldos de `balances.csv` se convierten con la tasa mediana de cada par de monedas.

**Contrapartes compartidas.** Solo 98 de 129.701 contrapartes aparecen en más de una empresa (91 en transacciones, 0 en facturas). Las 42.125 que cita otro equipo son las contrapartes que aparecen **a la vez en transacciones y en facturas**, no compartidas entre empresas. No hay red de clientes/proveedores compartidos.

**Saldo tras conversión.** El 20 % de empresas con saldo inconsistente no se debe a la moneda (23 % entre las que tienen cuentas en otra moneda, 19 % en el resto).